# Checkpoint B: ejecución del protocolo congelado

Este cuaderno solo lee los resultados de `run_checkpoint_b_frozen.py`. Las conclusiones son condicionales al supuesto no verificado de disponibilidad antes de abrir `t+1`; no hay HMM ni integración Actinver.

In [1]:
from pathlib import Path
import pandas as pd
ROOT=Path.cwd().resolve().parent
if not (ROOT/'reports').exists(): ROOT=Path.cwd().resolve()
R=ROOT/'reports'
summary=pd.read_csv(R/'checkpoint_b_partition_report.csv')
daily=pd.read_csv(R/'checkpoint_b_daily_results.csv',parse_dates=['signal_date','decision','breach'])
episodes=pd.read_csv(R/'checkpoint_b_episodes.csv',parse_dates=['start','end'])
display(summary)

,partition,coverage_denominator,skew_available,skew_missing,common_dates,episodes_evaluable,incomplete_episode_windows,bench_episodes,bench_recall,bench_alerts,...,bench_anticipation_mean,skew_episodes,skew_recall,skew_alerts,skew_frequency,skew_false_alarms,skew_false_alarm_rate,skew_precision,skew_anticipation_mean,decision
0,development,985,634,351,634,20,6,20,0.1,64,...,2.666667,20,0.1,64,0.063682,61,0.953125,0.046875,3.666667,NaN
1,validation,252,252,0,252,1,0,1,0.0,0,...,NaN,1,0.0,52,0.206349,52,1.000000,0.000000,NaN,NaN
2,final,406,406,0,406,4,0,4,0.0,23,...,NaN,4,0.0,76,0.187192,76,1.000000,0.000000,NaN,INCONCLUSO


## Verificaciones antes del desempeño

El script ejecuta pruebas sintéticas de: etiqueta con brecha futura, agrupación a cinco sesiones, asignación al primer episodio y exclusión por frontera.

**Límite.** Estas pruebas validan mecánica, no la validez económica del evento.

In [2]:
# Ejemplos concretos de etiqueta y episodio generados por la ejecución
display(daily.loc[daily['label'].eq(1), ['signal_date','decision','label','breach','skew','zskew']].head(5))
display(episodes.head(8))

,signal_date,decision,label,breach,skew,zskew
25,2020-02-07,2020-02-10,1,2020-02-25,0.05854,NaN
26,2020-02-10,2020-02-11,1,2020-02-25,0.05854,NaN
27,2020-02-11,2020-02-12,1,2020-02-25,0.05854,NaN
28,2020-02-12,2020-02-13,1,2020-02-25,0.04878,NaN
29,2020-02-13,2020-02-14,1,2020-02-25,0.04878,NaN


,start,end,part,start_idx
0,2020-02-25,2020-02-28,development,25
1,2020-03-09,2020-03-23,development,36
2,2020-04-01,2020-04-01,development,57
3,2020-06-11,2020-06-11,development,106
4,2020-09-08,2020-09-08,development,167
5,2020-10-28,2020-10-28,development,197
6,2022-01-19,2022-01-26,development,505
7,2022-02-18,2022-02-23,development,530


## Alertas, asignación y cobertura

Una alerta se asigna como máximo al primer episodio que inicia en las cinco decisiones posteriores. Fechas sin skew permanecen en el denominador de cobertura y equivalen a ausencia de alerta en una ventana de episodio.

**Límite.** La evaluación es de la definición congelada de caída de SPY, no de regímenes generales.

In [3]:
cols=['signal_date','part','bench_alert','skew_alert','wide','roll_or_jump']
display(daily.loc[(daily['bench_alert']) | (daily['skew_alert']), cols].head(12))
display(summary[['partition','coverage_denominator','skew_available','skew_missing','common_dates','episodes_evaluable','incomplete_episode_windows','decision']])

,signal_date,part,bench_alert,skew_alert,wide,roll_or_jump
40,2020-03-02,development,True,False,False,True
41,2020-03-03,development,True,False,False,True
42,2020-03-04,development,True,False,False,False
43,2020-03-05,development,True,False,False,False
44,2020-03-06,development,True,False,False,True
45,2020-03-09,development,True,False,False,True
46,2020-03-10,development,True,False,False,True
47,2020-03-11,development,True,False,False,False
48,2020-03-12,development,True,False,False,True
49,2020-03-13,development,True,False,False,False


,partition,coverage_denominator,skew_available,skew_missing,common_dates,episodes_evaluable,incomplete_episode_windows,decision
0,development,985,634,351,634,20,6,NaN
1,validation,252,252,0,252,1,0,NaN
2,final,406,406,0,406,4,0,INCONCLUSO


## Calidad y sensibilidad

La sensibilidad separada suprime alertas de skew en fechas con `wide_spread`, salto extremo o cambio de vencimiento; no elimina etiquetas ni redefine episodios.

**Límite.** Es una sensibilidad descriptiva y no puede reemplazar el resultado primario.

In [4]:
sens=pd.read_csv(R/'checkpoint_b_quality_sensitivity_daily.csv')
display(sens.groupby('part')[['wide','roll_or_jump','skew_alert']].sum())

,wide,roll_or_jump,skew_alert
part,,,
development,2,486,32
final,4,97,53
validation,2,61,39


## Decisión

La muestra final tiene menos de diez episodios evaluables, por lo que el estado obligatorio es **INCONCLUSO**. Métricas secundarias no pueden convertirlo en CANDIDATO ni STOP. Antes de cualquier ampliación, se debe mantener visible la dependencia del supuesto de disponibilidad de `t+1`.